In [103]:
import json
import os
import struct
import sys
from datetime import datetime
from pathlib import Path
from threading import Thread
from time import sleep

from IPython.display import HTML, clear_output, display
sys.path.append(str(Path("../../../scripts").resolve()))
from myradar import myradar
from parse_ti_mmwave_dat import parse_data

In [104]:
CFG_PATH = Path("xwr68xx_AOP.cfg")
OUTPUT_DIR = Path("..")
PATTERN = ["sit", "stand_up"]
FPS = 10
CHECK_DURATION = 1
PREPARE_TIME = 7
TRANSITION_TIME = 3

# to be edited
ACTION_TIME = 60
TOTAL_TIME = 140
USER_NAME = "lucas - lobby 3eme radar tres bas"
USER_SIZE = "176"
DISTANCE = "2"

# utils

In [105]:
def build_schedule(pattern, action_time, transition_time, total_time):
    partitions = int((total_time + transition_time) // (len(pattern) * (action_time + transition_time)))
    return pattern * partitions, partitions

In [106]:
schedule, partitions = build_schedule(PATTERN, ACTION_TIME, TRANSITION_TIME, TOTAL_TIME)
used_time = len(schedule) * ACTION_TIME + (len(schedule) - 1) * TRANSITION_TIME
print("partitions:", partitions)
print("blocks:", len(schedule))
print("used_time_s:", used_time)
print("total_time_s:", TOTAL_TIME)


partitions: 1
blocks: 2
used_time_s: 123
total_time_s: 140


In [107]:
def show_cue(label, seconds_left):
    key = label.lower().replace(" ", "_")
    bg = "#1f8a3b"
    blink = ""
    if key in ["stand_up", "standing_up"]:
        bg = "#1f4d8a"
    if key == "transition":
        bg = "#cc1f1f"
        blink = "animation: blink 1s steps(1,end) infinite;"
    if key == "get_ready":
        bg = "#e67e22"
        blink = "animation: blink 1s steps(1,end) infinite;"
    display(HTML(
        f"<style>@keyframes blink{{50%{{opacity:.45;}}}}</style>"
        f"<div style='height:260px;border-radius:18px;background:{bg};{blink}display:flex;flex-direction:column;justify-content:center;align-items:center;color:white'>"
        f"<div style='font-size:48px;font-weight:700;text-transform:capitalize'>{label}</div>"
        f"<div id='countdown-value' style='font-size:96px;line-height:1'>{seconds_left}</div>"
        f"</div>"
    ))


In [108]:
def run_countdown(label, duration_s):
    show_cue(label, duration_s)
    display(HTML(
        f"<script>(function(){{"
        f"const start=Date.now();"
        f"const duration={duration_s};"
        f"const el=document.getElementById('countdown-value');"
        f"if(!el) return;"
        f"const tick=()=>{{"
        f"const remaining=Math.max(0,duration-(Date.now()-start)/1000);"
        f"el.textContent=Math.max(1,Math.ceil(remaining));"
        f"if(remaining>0) requestAnimationFrame(tick);"
        f"}};tick();"
        f"}})();</script>"
    ))
    if label == "transition":
        for _ in range(duration_s):
            os.system("afplay /System/Library/Sounds/Ping.aiff >/dev/null 2>&1 &")
            sleep(1)
    else:
        sleep(duration_s)
    clear_output()


In [109]:
def start_radar_session(radar, cfg_path):
    radar.flush_data()
    radar.apply_cfg(str(cfg_path))
    sleep(0.1)


In [110]:
def capture_phase(radar, duration_s):
    return radar.read_stream(duration_s)


In [111]:
def parse_block(raw_bytes):
    frames = []
    cursor = 0
    magic = bytes((2, 1, 4, 3, 6, 5, 8, 7))
    while True:
        offset = raw_bytes.find(magic, cursor)
        if offset < 0 or offset + 40 > len(raw_bytes):
            break
        try:
            total_length = struct.unpack_from("<I", raw_bytes, offset + 12)[0]
        except struct.error:
            cursor = offset + 1
            continue
        if total_length < 40 or offset + total_length > len(raw_bytes):
            cursor = offset + 1
            continue
        packet = raw_bytes[offset : offset + total_length]
        try:
            parsed = parse_data(packet)
        except Exception:
            cursor = offset + 1
            continue
        if parsed:
            frames.extend(parsed)
        cursor = offset + total_length
    return frames


In [112]:
def add_block_label(frames, timeline, fps):
    labeled_frames = []
    phase_index = 0
    phase_start_s = 0
    phase_end_s = timeline[0][1] if timeline else 0
    for index, frame in enumerate(frames):
        session_time_s = index / fps
        while phase_index < len(timeline) - 1 and session_time_s >= phase_end_s:
            phase_index += 1
            phase_start_s = phase_end_s
            phase_end_s += timeline[phase_index][1]
        label = timeline[phase_index][0] if timeline else None
        frame["block_time_s"] = session_time_s - phase_start_s
        frame["session_time_s"] = session_time_s
        if label is not None:
            frame["label"] = label
        labeled_frames.append(frame)
    return labeled_frames


In [113]:
def run_session(radar, cfg_path, schedule, action_time, transition_time, prepare_time, fps):
    timeline = []
    for block_index, label in enumerate(schedule):
        timeline.append((label, action_time))
        if block_index < len(schedule) - 1:
            timeline.append((None, transition_time))
    total_duration = sum(duration_s for _, duration_s in timeline)
    run_countdown("get_ready", prepare_time)
    start_radar_session(radar, cfg_path)
    raw_data = []
    capture_thread = Thread(target=lambda: raw_data.append(capture_phase(radar, total_duration)))
    capture_thread.start()
    try:
        for label, duration_s in timeline:
            run_countdown("transition" if label is None else label, duration_s)
        capture_thread.join()
    finally:
        radar.sendcmd("sensorStop")
    raw_bytes = raw_data[0]
    frames = parse_block(raw_bytes)
    merged_frames = add_block_label(frames, timeline, fps)
    print("timeline_duration_s:", total_duration)
    print("total_raw_bytes:", len(raw_bytes))
    print("total_frames:", len(merged_frames))
    return merged_frames


In [114]:
def save_labeled_frames(path, frames, record_info):
    path.write_text(json.dumps({"record": record_info, "frames": frames}, indent=2))


In [115]:
def build_record_output(output_dir, duration, user_name, user_size, distance):
    record_ids = []
    for path in output_dir.glob("record*"):
        number = ""
        for char in path.name[6:]:
            if char.isdigit():
                number += char
            else:
                break
        if number:
            record_ids.append(int(number))
    next_id = max(record_ids) + 1 if record_ids else 1
    record_name = f"record{next_id}"
    date = datetime.now().astimezone().isoformat(timespec="minutes")
    output_path = output_dir / f"{record_name}.labeled.json"
    record_info = {
        "name": record_name,
        "duration": duration,
        "date": date,
        "user_name": user_name,
        "user_size": user_size,
        "distance": distance,
    }
    return output_path, record_info

In [116]:
SESSION_DURATION = len(schedule) * ACTION_TIME + (len(schedule) - 1) * TRANSITION_TIME
OUTPUT_PATH, RECORD_INFO = build_record_output(OUTPUT_DIR, SESSION_DURATION, USER_NAME, USER_SIZE, DISTANCE)
print(OUTPUT_PATH.name)
print(RECORD_INFO)

record22.labeled.json
{'name': 'record22', 'duration': 123, 'date': '2026-05-12T10:42+02:00', 'user_name': 'lucas - lobby 3eme radar tres bas', 'user_size': '176', 'distance': '2'}


In [117]:
radar = myradar(verbose=False)
print(radar._config_port, radar._data_port)

/dev/tty.usbserial-00D20BC30 /dev/tty.usbserial-00D20BC31


In [118]:
def check_radar_ready(radar):
    ready = radar.is_connected()
    print("check_connected:", ready)
    return ready

In [119]:
radar_ready = check_radar_ready(radar)
print("radar_ready:", radar_ready)

check_connected: True
radar_ready: True


# recorder

In [120]:
merged_frames = run_session(radar, CFG_PATH, schedule, ACTION_TIME, TRANSITION_TIME, PREPARE_TIME, FPS)
print(len(merged_frames))

timeline_duration_s: 123
total_raw_bytes: 926447
total_frames: 1230
1230


In [121]:
print(merged_frames[0]["label"], merged_frames[0]["session_time_s"])
print(merged_frames[-1]["label"], merged_frames[-1]["session_time_s"])

sit 0.0
stand_up 122.9


In [122]:
save_labeled_frames(OUTPUT_PATH, merged_frames, RECORD_INFO)
print(OUTPUT_PATH)

../record22.labeled.json


In [123]:
def close_radar(radar):
    radar.close()

In [124]:
close_radar(radar)
print("radar closed")

radar closed
